In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)


# 6.3 train/val/test 분리 원칙 실전 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter06_3_train_val_test.ipynb)

책 본문: [6.3 train/val/test 분리 원칙 실전](https://smhanlab.com/book-ml/kor/ml1/chapter06/3.html)

이 노트북은 책 6.3절의 내용을 코드로 재현합니다: (1) 3분할 원칙에서 **정보 흐름**이 어디로만 가야 하는지, (2) 테스트를 "살짝 훔쳐보며" 하이퍼파라미터를 고르는 **선택 편향(selection bias)**이 통계적으로 얼마나 낙관적인지, (3) **정규화 통계량 누수**가 z-값을 어떻게 왜곡하는지, (4) 의료 영상에서 실제로 반복되는 **환자 단위 누수**(무작위 분할 vs 그룹 분할)가 성능을 얼마나 부풀리는지, (5) **시계열 데이터**에서 무작위 분할이 왜 붕괴하는지, (6) diabetes 데이터에서 **3분할 파이프라인 전체**로 \(\lambda\)를 검증셋으로만 골라 최종 성능을 한 번 재는 흐름까지 봅니다. numpy/scikit-learn만 씁니다.

**핵심 원칙** — 학습 데이터로 파라미터 \(w\), 검증 데이터로 하이퍼파라미터(\(\lambda, k\) 등), **테스트 데이터는 최종 성능만 딱 한 번**. 테스트는 어떤 단계에도 "두 번째"로 다시 쓰면 안 된다 — 테스트 정보가 학습이나 튜닝에 스며들면(= data leakage) 보고된 성능은 실제보다 낙관적이다. 모든 시드는 고정되어 있어서 결과 재현이 가능하다.

## 1. 3분할: 정보 흐름의 방향

"train으로 \(w\)를, val로 하이퍼파라미터를, test로 최종 성능"이라는 원칙은 사실 **정보의 흐름 방향**을 정한 규칙이다. 아래 다이어그램(본문에 `ch06_3_info_flow.svg`으로 삽입됨)처럼, 정보가 흐르는 허용된 방향은 왼쪽(train)→가운데(val)→아래(최종 모델)로만 되고, **테스트(빨간 점선)에서 다른 곳으로 정보가 흐르는 모든 길은 금지**다.

- train → val/test: 전처리 통계량(평균·표준편차, PCA 축 등)을 **train으로만 fit**하고 val/test에는 그걸 **apply**하는 방향.
- train → model: \(w\)를 fit.
- val → model: 하이퍼파라미터를 고르는 방향.
- test → (val/train/model): 금지. 테스트를 보고 하이퍼파라미터를 다시 고르거나, 테스트를 다시 train에 넣거나, 분할을 바꿀 때 테스트를 기준으로 움직이는 것 — 전부 "살짝 훔쳐보기"다.

이 그림은 코드 없이도 이해가 되지만, 뒤의 §2~§6에서는 "이 금지된 흐름을 한 번씩 밟아봤을 때 성능 숫자가 실제로 어떻게 바뀌는가"를 하나씩 보여준다.

## 2. "살짝 훔쳐보기"의 통계: 선택 편향(selection bias)

검증 대신 **테스트 데이터로 여러 후보를 돌려보고 가장 잘 나온 걸 고르면** 어떻게 되는가? 각 후보의 테스트 성능은 (진짜 성능 + 우연한 잡음)으로 모델링할 수 있다. 잡음이 독립적 정규분포 \(\mathcal{N}(0,\sigma^2)\)라면, 후보 \(m\)개 중 **최고 점**을 고르는 행위는 \(m\)번의 잡음 뽑기 중 max를 고르는 것과 같다 — max는 평균(=0, 진짜 성능)보다 **항상 위쪽**에 치우친다.

아래에서 \(\sigma=1\)(잡음 크기를 임의의 1로)이고 진짜 성능이 0인 상황을 4만 번 시뮬레이션해, 후보 수 \(m\)에 따라 "고른 최고 점"의 평균 \(\mathbb{E}[\max]\)을 재본다. 이게 **테스트 성능의 기대 오차**다 — 후보가 많을수록 테스트는 실제보다 낙관적으로 읽힌다.

In [2]:
import numpy as np

def e_max(m, sigma, n_sims=40000, seed=1):
    g = np.random.default_rng(seed)
    return g.normal(0, sigma, size=(n_sims, m)).max(axis=1).mean()

print("후보 수 m   E[max] (진짜 성능 0, 잡음 sigma=1)")
for m in [1, 2, 5, 10, 20, 50, 100, 500]:
    print(f"  m={m:4d}   {e_max(m, 1.0):.3f}   <- 테스트로 고르면 이렇게 부풀어 오른다")


후보 수 m   E[max] (진짜 성능 0, 잡음 sigma=1)
  m=   1   -0.009   <- 테스트로 고르면 이렇게 부풀어 오른다
  m=   2   0.561   <- 테스트로 고르면 이렇게 부풀어 오른다
  m=   5   1.160   <- 테스트로 고르면 이렇게 부풀어 오른다
  m=  10   1.532   <- 테스트로 고르면 이렇게 부풀어 오른다
  m=  20   1.863   <- 테스트로 고르면 이렇게 부풀어 오른다
  m=  50   2.247   <- 테스트로 고르면 이렇게 부풀어 오른다
  m= 100   2.507   <- 테스트로 고르면 이렇게 부풀어 오른다


  m= 500   3.036   <- 테스트로 고르면 이렇게 부풀어 오른다


In [3]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ms = [1, 2, 5, 10, 20, 50, 100, 500]
emax = [e_max(m, 1.0) for m in ms]
ax.semilogx(ms, emax, "o-", color="#1d4ed8", lw=2)
ax.axhline(0, color="gray", ls=":", lw=1)
ax.axhline(1.0, color="#dc2626", ls="--", lw=1.2, label="One unit of noise ($\\sigma$=1)")
ax.set_xlabel("Number of candidates $m$ compared on the test set (log)")
ax.set_ylabel("$\\mathbb{E}[\\max]$ — the amount of inflation")
ax.set_title("Selecting candidates on the test set: more candidates, more optimistic")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(IMG + "/ch06_3_selection_bias.svg")
plt.show()


읽어볼 점:

- \(m=1\)이면 \(\mathbb{E}[\max] \approx 0\) — 후보 한 개만 보면 편향이 없다(진짜 성능 0).
- \(m=5\)(λ 후보 5개)만 되도 \(\approx 1.16\), \(m=50\)이면 \(\approx 2.25\) — **잡음 하나(\(\sigma=1\))의 2배** 이상 테스트가 부풀어 오른다.
- 실전에서는 \(m\)이 "모델 수 × 하이퍼파라미터 수"의 곱으로 커지기 쉽다(모델 5개 × λ 20개 = 100개 후보 → 부풀림 \(\approx 2.5\)).

이게 "테스트로 모델을 고르면 안 된다"는 원칙의 정량적 이유다. 검증셋으로 고르면 검증셋만 이 편향을 먹고, **테스트는 한 번만, 마지막에** 쓰이므로 부풀림이 테스트에 전이되지 않는다. (참고: 이 표의 \(\sigma\)는 잡음의 절대 크기가 아니라 "성능 측정의 흔들림"을 표준화한 값으로, 실제에서는 \(\sigma\)를 곱하면 된다 — 잡음이 3배면 부풀림도 3배.)

## 3. 데이터 누수: 정규화 통계량을 "전체 데이터"로 fit하면?

본문의 "손으로 한 번" 예(1~10을 train 7개 / test 3개로 나눔)를 코드로 확인한다. 표준화(스케일링)는 **train으로만 fit**하고 test에는 apply하는 것이 배포 상황을 흉내 낸 것이다. 전체 데이터(= train+test)로 fit하면 test의 정보가 전처리에 스민다.

두 경우 모두 test \([8,9,10]\)의 z-값을 계산해 비교한다: **누수(전체 fit)**는 test 값을 "평범한"(z가 작은) 것처럼 보이게 하고, **올바른(train만 fit)**는 test 값이 실제로는 학습 범위 밖(큰 z)임을 드러낸다.

In [4]:
import numpy as np

data = np.arange(1, 11, dtype=float)   # 1 .. 10
tr, te = data[:7], data[7:]            # train = 1..7, test = 8..10

# ddof=0 (population std) — 본문의 표와 일치
def standardize(train_stats_source, x):
    mu, sd = train_stats_source.mean(), train_stats_source.std(ddof=0)
    return (x - mu) / sd, mu, sd

z_leak, mu_l, sd_l = standardize(data, te)   # WRONG: fit on ALL (train+test)
z_ok,   mu_o, sd_o = standardize(tr, te)     # RIGHT: fit on train only

print(f"전체 데이터(누수):  mean={mu_l:.3f}  std={sd_l:.3f}  -> test z = {np.round(z_leak, 3)}")
print(f"train만(올바름):    mean={mu_o:.3f}  std={sd_o:.3f}  -> test z = {np.round(z_ok, 3)}")
print()
print("test 값 [8, 9, 10]이 '학습 범위(1~7)' 밖이라는 사실:")
print(f"  누수 통계량으로 보면 z = {z_leak[0]:.2f}~{z_leak[-1]:.2f} (평범해 보임)")
print(f"  train 통계량으로 보면 z = {z_ok[0]:.2f}~{z_ok[-1]:.2f} (극단적으로 큼 = OOD)")


전체 데이터(누수):  mean=5.500  std=2.872  -> test z = [0.87  1.219 1.567]
train만(올바름):    mean=4.000  std=2.000  -> test z = [2.  2.5 3. ]

test 값 [8, 9, 10]이 '학습 범위(1~7)' 밖이라는 사실:
  누수 통계량으로 보면 z = 0.87~1.57 (평범해 보임)
  train 통계량으로 보면 z = 2.00~3.00 (극단적으로 큼 = OOD)


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# (왼쪽) 원본 값: train(1~7) vs test(8~10)
ax = axes[0]
ax.scatter(tr, [0]*len(tr), s=90, color="#1d4ed8", label="train (1~7)", zorder=3)
ax.scatter(te, [0]*len(te), s=90, marker="^", color="#dc2626", label="test (8~10)", zorder=3)
ax.axvline(mu_o, color="#1d4ed8", ls="--", lw=1.2)
ax.text(mu_o+0.05, 0.15, f"train mean={mu_o:.1f}", fontsize=8, color="#1d4ed8")
ax.set_ylim(-0.6, 0.6); ax.set_xlim(0.2, 10.8)
ax.set_xlabel("Original value"); ax.set_yticks([])
ax.set_title("Original axis: test lies outside the train range (1-7)")
ax.legend(fontsize=8, loc="lower right")

# (오른쪽) z-값을 두 통계량으로
ax = axes[1]
xs = np.linspace(-0.2, 3.6, 300)
ax.axvspan(0, 0.2, color="lightgray", alpha=0.5)
ax.scatter(z_ok, [1]*len(z_ok), s=90, marker="^", color="#16a34a", label=f"train-only fit (correct): z=2.0, 2.5, 3.0", zorder=3)
ax.scatter(z_leak, [0]*len(z_leak), s=90, marker="v", color="#dc2626", label=f"full-data fit (leakage): z={z_leak[0]:.2f}~{z_leak[-1]:.2f}", zorder=3)
ax.set_ylim(-0.6, 1.6); ax.set_xlim(-0.2, 3.6)
ax.set_yticks([0,1]); ax.set_yticklabels(["full-data fit (leakage)", "train-only fit"])
ax.set_xlabel("z-score of the test values")
ax.set_title("Same test, different statistics → different z-scores")
ax.legend(fontsize=8, loc="upper left")

fig.suptitle("Normalization-statistics leakage: when std grows because of the test set, the test looks 'ordinary'", fontsize=11)
fig.tight_layout()
fig.savefig(IMG + "/ch06_3_leakage_zscore.svg")
plt.show()


왜 std가 다른가 — 전체 10개의 std(2.872)는 **test의 큰 값들(8,9,10) 때문에 이미 커져 있다.** 그래서 같은 test 값을 전체 통계로 나누면 z가 작아("평범") 보이고, train만(1~7)의 std(2.0)으로 나누면 z가 크게(극단) 보인다. 오른쪽 그림에서 보듯, **test는 실제로 학습 범위 밖(큰 z)의 새로운 데이터**인데, 누수 통계량으로 표준화하면 "평범한" 것처럼 위장한다. 배포 시점에는 "미래에 들어올 데이터(test)"를 볼 수 없으므로 train 통계량만 쓰는 것이 유일한 정직한 방법이다 — 이 원칙은 스케일링뿐 아니라 PCA의 주성분(Chapter 14), 결측치 대체 평균 등 **데이터를 보고 계산하는 모든 통계량**에 적용된다. (scikit-learn에서 `StandardScaler().fit(X_train)` 뒤 `transform(X_test)`이 이 원칙의 표준 구현이다.)

## 4. 환자 단위 누수: 무작위 분할 vs 그룹(환자) 분할

본문의 "유명한 사례"를 숫자로 재현한다. 환자 \(P=100\)명, 각 환자당 스캔 \(T=5\)장. 각 환자는 자기만의 "시그니처" 위치(center)에 5장의 이미지가 매우 가깝게(잡음 0.1) 찍혀 있고, 환자에게는 하나의 연속값(예: 혈압)이 있다. **이미지 위치만으로 환자를 구별할 수 있지만, 환자와 값의 실제 관계는 없다** — 관계가 없기 때문에 "새로운 환자"를 정확히 예측하는 것은 애초에 불가능하다.

- **무작위 분할**(500장 이미지를 섞어 80/20): 같은 환자의 일부 이미지가 train에, 일부가 test에 흩어진다. kNN은 test 이미지의 k=1 이웃을 **같은 환자의 train 이미지**로 찾아 환자를 "인식"하고 값을 정확히 재현한다 → RMSE가 비현실적으로 낮아진다(과대평가).
- **환자(그룹) 분할**: 환자의 5장이 통째로 train 또는 test 중 하나로만 간다. test 환자는 train에서 **한 번도 안 본** 환자이므로 kNN은 인식이 안 되고, 환자를 예측하는 한계(대체로 집단 평균 근처)에 갇힌다 → RMSE가 실제 일반화 성능을 보인다.

두 분할의 test RMSE 차이가 바로 **환자 단위 누수가 만든 "환상"** 이다.

In [6]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

P, T = 100, 5
rng = np.random.default_rng(51)
v = rng.normal(0, 2.0, size=P)                 # 환자의 값(표적), sigma=2
center = rng.normal(0, 2.0, size=(P, 2))       # 환자별 시그니처(위치가 환자마다 다름)
X = (center[:, None, :] + rng.normal(0, 0.1, size=(P, T, 2))).reshape(P * T, 2)
y = np.repeat(v, T)
pat = np.repeat(np.arange(P), T)               # 각 이미지가 어느 환자인지

def knn_rmse(Xtr, ytr, Xte, yte):
    m, s = Xtr.mean(0), Xtr.std(0) + 1e-9
    pred = KNeighborsRegressor(n_neighbors=1).fit((Xtr - m) / s, ytr).predict((Xte - m) / s)
    return float(np.sqrt(mean_squared_error(yte, pred)))

# (a) 무작위 분할 (이미지 단위) — 같은 환자가 train/test에 섞임
perm = np.random.default_rng(52).permutation(len(X))
cut = int(len(X) * 0.8)
rmse_rand = knn_rmse(X[perm[:cut]], y[perm[:cut]], X[perm[cut:]], y[perm[cut:]])

# (b) 환자(그룹) 분할 — 환자의 5장이 통째로
gperm = np.random.default_rng(53).permutation(P)
tr_pat, te_pat = gperm[:80], gperm[80:]
tr_rows = np.concatenate([np.arange(p * T, p * T + T) for p in tr_pat])
te_rows = np.concatenate([np.arange(p * T, p * T + T) for p in te_pat])
rmse_grp = knn_rmse(X[tr_rows], y[tr_rows], X[te_rows], y[te_rows])

print(f"(a) 무작위 분할(이미지 단위):  test RMSE = {rmse_rand:.3f}   <- 비현실적으로 낮음 (환자 인식=누수)")
print(f"(b) 환자(그룹) 분할:          test RMSE = {rmse_grp:.3f}   <- 실제 일반화 성능")
print(f"    환상(과대평가) = {rmse_grp - rmse_rand:+.3f}")
print(f"    (참고: 새로운 환자를 예측하는 한계 ~ sigma_v*sqrt(2) ~ {2*np.sqrt(2):.2f}, 즉 무작위 분할은 이 한계를 '달성한 척' 한다)")


(a) 무작위 분할(이미지 단위):  test RMSE = 1.042   <- 비현실적으로 낮음 (환자 인식=누수)
(b) 환자(그룹) 분할:          test RMSE = 3.651   <- 실제 일반화 성능
    환상(과대평가) = +2.608
    (참고: 새로운 환자를 예측하는 한계 ~ sigma_v*sqrt(2) ~ 2.83, 즉 무작위 분할은 이 한계를 '달성한 척' 한다)


In [7]:
# 환자=클러스터 시각화: 5장의 이미지는 같은 환자의 blob에 뭉침
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
rng_show = np.random.default_rng(52)   # (a)와 같은 무작위 분할
tr_set, te_set = set(perm[:cut]), set(perm[cut:])
for i in range(len(X)):
    col_tr = "#1d4ed8" if i in tr_set else "#dc2626"
    axes[0].scatter(X[i,0], X[i,1], c=col_tr, s=26, alpha=0.8, edgecolors="k", linewidths=0.3)
axes[0].set_title("(a) Random split: train (blue) / test (red) mixed within each blob", fontsize=10)
axes[0].set_xlabel("x1"); axes[0].set_ylabel("x2"); axes[0].set_aspect("equal")

gshow = np.random.default_rng(53).permutation(P)
te_pat_show = set(gshow[80:])
for p in range(P):
    col = "#dc2626" if p in te_pat_show else "#1d4ed8"
    rows = np.arange(p*T, p*T+T)
    axes[1].scatter(X[rows,0], X[rows,1], c=col, s=26, alpha=0.8, edgecolors="k", linewidths=0.3)
axes[1].set_title("(b) Patient split: each blob goes wholly to one side", fontsize=10)
axes[1].set_xlabel("x1"); axes[1].set_ylabel("x2"); axes[1].set_aspect("equal")
from matplotlib.lines import Line2D
handles = [Line2D([0],[0], marker="o", color="w", markerfacecolor="#1d4ed8", markersize=7, label="train"),
           Line2D([0],[0], marker="o", color="w", markerfacecolor="#dc2626", markersize=7, label="test")]
fig.legend(handles=handles, loc="lower center", ncol=2, fontsize=9)
fig.suptitle("Patient-level leakage: the 5 images of the same patient cluster into one blob", fontsize=11)
fig.tight_layout(rect=[0, 0.05, 1, 0.96])
fig.savefig(IMG + "/ch06_3_patient_leakage.svg")
plt.show()


무작위 분할(왼쪽)에서는 **매 blob 안**이 파랑/빨강으로 섞여 있다 — kNN이 test 점의 이웃을 찾으면 같은 blob(같은 환자)의 train 점이 항상 근처에 있어 환자를 "인식"한다. 환자 분할(오른쪽)에서는 blob이 통째로 한 쪽이므로, test blob의 환자는 train에서 한 번도 안 봤다. 무작위 분할 RMSE(\(\approx1.0\))가 환자별 값의 표준편차(2.0)보다 훨씬 낮고, 새로운 환자를 예측하는 이론 한계(\(\sigma\sqrt{2}\approx2.8\)) 아래로 내려간 이유는 **모델이 관계를 배운 것이 아니라 환자를 외운 것**이다. 실전에서 `sklearn.model_selection.GroupKFold`(그룹=환자)를 써야 하는 이유가 바로 여기 — "실전에서 실제로 겹칠 수 없는 단위"를 그룹 키로 쓰는 것이다.

## 5. 시계열: 무작위 분할은 왜 붕괴하는가

환자 누수와 같은 "단위" 문제의 다른 얼굴이 **시계열**이다. 시간 순서대로 증가하는 추세(매일 +1)가 있는 일일 데이터(365일)를 만든다. 특징은 **시간 \(t\)뿐**(표적은 포함 안 함 — 표적을 특징에 넣으면 그 자체가 또 다른 누수).

- **무작위 분할**: test 점들이 시간축 곳곳에 흩어진다. kNN(\(k=5\))은 test 점의 시간 근처에 **train 점**이 항상 있으므로(과거/미래 가리지 않고) 잘 맞힌다 → RMSE가 낮아 보이지만, **미래를 과거로 예측한 것**이라 배포에서는 성립하지 않는다.
- **시간순 분할**: 앞에서 80%를 train, 뒤의 73일을 test. test는 train의 시간 범위(**horizon**)를 벗어나므로 kNN은 밖으로 외삽하게 되고, 추세를 못 쫓아 RMSE가 급등한다.

이 두 RMSE의 차이가 "시계열은 무작위가 아니라 **시간순으로** 나눠야 한다"는 원칙의 정량적 이유다.

In [8]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(71)
t = np.arange(0, 365, dtype=float)
y = 100.0 + 1.0 * t + rng.normal(0, 0.5, size=len(t))   # 강한 상승 추세 + 잡음
X = t.reshape(-1, 1)                                     # 특징 = 시간만

def knn_rmse(Xtr, ytr, Xte, yte):
    m, s = Xtr.mean(0), Xtr.std(0) + 1e-9
    pred = KNeighborsRegressor(n_neighbors=5).fit((Xtr - m) / s, ytr).predict((Xte - m) / s)
    return float(np.sqrt(mean_squared_error(yte, pred)))

# (a) 무작위 분할
perm = np.random.default_rng(72).permutation(len(t))
cut = int(len(t) * 0.8)
rmse_rand = knn_rmse(X[perm[:cut]], y[perm[:cut]], X[perm[cut:]], y[perm[cut:]])

# (b) 시간순 분할 (뒤 20% = test, train horizon 밖)
c2 = int(len(t) * 0.8)
rmse_chrono = knn_rmse(X[:c2], y[:c2], X[c2:], y[c2:])

print(f"(a) 무작위 분할:  test RMSE = {rmse_rand:.3f}   <- train 점 사이에 test가 숨어 있음 (미래를 과거로)")
print(f"(b) 시간순 분할:  test RMSE = {rmse_chrono:.3f}   <- train horizon 밖 외삽, 실제 성능")
print(f"    (추세 기울기 1/일, test는 train 마지막 이후 {len(t)-c2}일을 커버)")


(a) 무작위 분할:  test RMSE = 0.808   <- train 점 사이에 test가 숨어 있음 (미래를 과거로)
(b) 시간순 분할:  test RMSE = 44.117   <- train horizon 밖 외삽, 실제 성능
    (추세 기울기 1/일, test는 train 마지막 이후 73일을 커버)


시계열에서는 분할 방향이 "무작위 vs 시간순"을 넘어 **데이터의 물리적 순서**를 존중하는 문제다. 로그, 주가, 환자 반복 측정, IoT 센서 등 "시간/순서가 의미 있는" 데이터에서는 `train_test_split`의 `shuffle=False`(시간순)나 `TimeSeriesSplit`을 쓰고, 그룹이 있는 데이터에서는 `GroupKFold`/`GroupShuffleSplit`을 쓰는 것이 표준이다 — 6.2절의 `k_fold_split`이 순서대로 자르는 것과 같은 "분할이 구조를 깨면 안 된다"는 원칙의 연장선이다.

## 6. diabetes에서 3분할 파이프라인 전체

마지막으로 이 절의 원칙을 **한 흐름**으로 묶는다. scikit-learn의 `load_diabetes`(442건, 10개 특징)을 쓰고:

1. **한 번에** 60/20/20처럼 3분할하지 않고, 80/20 → (train 50% / val+test 50% → val/test 절반씩)의 **2단계 분할**로 `train / val / test`를 만든다(아래 §7에서 이 2단계가 pairwise disjoint인지 확인).
2. \(\lambda\) 후보를 돌려 **val RMSE로만** 최적 \(\lambda\)를 고른다(§2의 선택 편향이 val에서만 발생).
3. 그 \(\lambda\)로 train으로만 다시 fit하고, **test RMSE를 딱 한 번** 잰다.
4. 대조: 만약(잘못) test RMSE로 \(\lambda\)를 골랐다면 어떤 \(\lambda\)를 고르게 되는지 — test가 튜닝에 쓰이면 "그 test에서 제일 좋은 값"을 골라버려, 이 test의 수치는 더 이상 정직한 일반화 추정치가 아님을 보여준다.

In [9]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.datasets import load_diabetes

data = load_diabetes()
Xdd, ydd = data.data, data.target
print("diabetes:", Xdd.shape, " (표적 범위 %.0f ~ %.0f)" % (ydd.min(), ydd.max()))

# 2단계 3분할: 전체의 40%를 (val+test)로 떼고, 그 40%를 다시 절반씩
Xtr, Xtemp, ytr, ytemp = train_test_split(Xdd, ydd, test_size=0.4, random_state=0)
Xva, Xte, yva, yte = train_test_split(Xtemp, ytemp, test_size=0.5, random_state=0)
print(f"크기: train={len(Xtr)}  val={len(Xva)}  test={len(Xte)}")

lams = [0.1, 1.0, 10.0, 100.0, 1000.0]
val_rmse = {lam: float(np.sqrt(mean_squared_error(yva, Ridge(alpha=lam).fit(Xtr, ytr).predict(Xva)))) for lam in lams}
print("val RMSE per lambda:", {k: round(v, 2) for k, v in val_rmse.items()})
best_v = min(val_rmse, key=val_rmse.get)
final_test = float(np.sqrt(mean_squared_error(yte, Ridge(alpha=best_v).fit(Xtr, ytr).predict(Xte))))
print(f"val로 고른 lambda = {best_v}  ->  최종 test RMSE (한 번만) = {final_test:.2f}")

# 대조: (잘못) test로 lambda 고르기
test_rmse = {lam: float(np.sqrt(mean_squared_error(yte, Ridge(alpha=lam).fit(Xtr, ytr).predict(Xte)))) for lam in lams}
best_t = min(test_rmse, key=test_rmse.get)
print(f"(잘못) test로 고르면 lambda = {best_t}  ->  그때 test RMSE = {test_rmse[best_t]:.2f}")
print("  -> test로 고른 값은 '그 test에서 우연히 제일 좋은 것'이라 더 이상 정직한 일반화 추정치가 아님")


diabetes: (442, 10)  (표적 범위 25 ~ 346)
크기: train=265  val=88  test=89
val RMSE per lambda: {0.1: 58.07, 1.0: 61.75, 10.0: 73.04, 100.0: 77.49, 1000.0: 78.07}
val로 고른 lambda = 0.1  ->  최종 test RMSE (한 번만) = 53.79
(잘못) test로 고르면 lambda = 1.0  ->  그때 test RMSE = 53.06
  -> test로 고른 값은 '그 test에서 우연히 제일 좋은 것'이라 더 이상 정직한 일반화 추정치가 아님


주의: val로 고른 \(\lambda\)와 test로 고른 \(\lambda\)가 **같을 수도, 다를 수도 있다** — 이 작은 데이터에서는 둘 다를 "정답처럼" 보여서 더 위험하다. 결정적인 차이는 **절차**다: val로 고르면 test는 아직 한 번도 안 본 "처음 보는 데이터"로 남아 최종 수치를 정직하게 재주지만, test로 고르면 test 자체가 선택에 쓰였으므로 이 test의 수치는 §2의 선택 편향을 그대로 먹고 있는, 부풀려진 추정치다. 데이터가 커질수록 이 차이가 더 크게 벌어진다(§2의 \(\mathbb{E}[\max]\)가 후보 수에 따라 커지듯).

## 7. 확인: 2단계 3분할은 정말 pairwise disjoint인가

§6의 2단계 분할(`train_test_split` 두 번)이 실제로 **서로 안 겹치는** train/val/test를 만드는지, val+test가 정확히 "한 번에 40%를 떼어둔" 부분과 일치하는지 unique id로 검증한다 — "val/test가 처음에 고른 40% 안에 들어 있는지"가 3분할 원칙의 기계적 점검 항목이다.

In [10]:
import numpy as np
from sklearn.model_selection import train_test_split

Xid = np.arange(1000).reshape(-1, 1)
Xtr_i, Xt_i, _, _ = train_test_split(Xid, np.zeros(1000), test_size=0.4, random_state=0)
Xva_i, Xte_i, _, _ = train_test_split(Xt_i, np.zeros(400), test_size=0.5, random_state=0)
s_tr, s_va, s_te = set(map(int, Xtr_i.ravel())), set(map(int, Xva_i.ravel())), set(map(int, Xte_i.ravel()))

print(f"크기: train={len(s_tr)}  val={len(s_va)}  test={len(s_te)}")
print("pairwise disjoint:",
      s_tr.isdisjoint(s_va), s_tr.isdisjoint(s_te), s_va.isdisjoint(s_te))
print("val+test가 처음 고른 40%와 일치:",
      len(s_va | s_te) == 400 and (s_va | s_te).isdisjoint(s_tr))


크기: train=600  val=200  test=200
pairwise disjoint: True True True
val+test가 처음 고른 40%와 일치: True
